In [2]:
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
df= pd.read_csv("data/processed_rat_data.csv")

In [ ]:
df.info()

1. FCR Over Time for Each Water Group

What we're measuring:

- FCR (Feed Conversion Ratio) = weekly_feed_intake / weekly_weight_gain
- Lower FCR = more efficient (less feed needed per gram of weight gain)
- Higher FCR = less efficient (more feed needed for same weight gain)

Key questions:

- Which water group maintains the best (lowest) FCR throughout the study?
- At what week does FCR start deteriorating for each group?
- Is there a "plateau week" where FCR suddenly spikes?

In [3]:
# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Create output directory
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

print(f"Data loaded: {len(df)} rows")
print(f"Unique rats: {df['unique_rat_id'].nunique()}")
print(f"Water groups: {df['water_group'].nunique()}")

# ============================================================================
# STEP 1: Calculate FCR (Feed Conversion Ratio)
# ============================================================================
print("\n" + "="*80)
print("STEP 1: Calculating FCR")
print("="*80)

# Option 1: Exclude weeks with weight gain < 2g
WEIGHT_GAIN_THRESHOLD = 2.0

df['fcr'] = df['weekly_feed_intake'] / df['weekly_weight_gain']

# Create a clean version excluding low weight gain weeks
df_clean = df[df['weekly_weight_gain'] >= WEIGHT_GAIN_THRESHOLD].copy()

print(f"\nOriginal rows: {len(df)}")
print(f"After excluding weight_gain < {WEIGHT_GAIN_THRESHOLD}g: {len(df_clean)}")
print(f"Rows excluded: {len(df) - len(df_clean)} ({100*(len(df) - len(df_clean))/len(df):.1f}%)")

# Option 3: Calculate inverse metric (Efficiency Score) as validation
df['efficiency_score'] = (df['weekly_weight_gain'] / df['weekly_feed_intake']) * 100
df_clean['efficiency_score'] = (df_clean['weekly_weight_gain'] / df_clean['weekly_feed_intake']) * 100

# Show FCR statistics by week
print("\n--- FCR Statistics by Week (Clean Data) ---")
fcr_by_week = df_clean.groupby('week')['fcr'].agg(['mean', 'median', 'std', 'count'])
print(fcr_by_week)

# Show which weeks were most affected by exclusion
print("\n--- Weeks Most Affected by <2g Exclusion ---")
exclusion_by_week = df.groupby('week').size() - df_clean.groupby('week').size()
print(exclusion_by_week.sort_values(ascending=False).head(10))

# ============================================================================
# STEP 2: Prepare data for visualization
# ============================================================================
print("\n" + "="*80)
print("STEP 2: Preparing visualization data")
print("="*80)

# Calculate summary statistics by water group and week
fcr_summary = df_clean.groupby(['water_group', 'week']).agg({
    'fcr': ['mean', 'median', 'std', 'count'],
    'efficiency_score': ['mean', 'median']
}).reset_index()

# Flatten column names
fcr_summary.columns = ['_'.join(col).strip('_') for col in fcr_summary.columns.values]
fcr_summary.rename(columns={
    'water_group': 'water_group',
    'week': 'week'
}, inplace=True)

# Create phases for box plots
def assign_phase(week):
    if week <= 5:
        return 'Early (1-5)'
    elif week <= 10:
        return 'Middle (6-10)'
    else:
        return 'Late (11-15)'

df_clean['phase'] = df_clean['week'].apply(assign_phase)

# Create shortened water group names for better visualization
def shorten_group_name(name):
    if 'Group 1' in name and 'RO water with < 20' in name:
        return 'G01: RO <20 TDS'
    elif 'Group 2' in name:
        return 'G02: RO 50-75 TDS'
    elif 'Group 3' in name and 'alternate' not in name.lower():
        return 'G03: RO 125-150 TDS'
    elif 'Group 4' in name:
        return 'G04: Telugu Ganga'
    elif 'Group 5' in name and 'alternate' not in name.lower():
        return 'G05: Kalyani Dam'
    elif 'Group 6' in name and 'alternate' not in name.lower():
        return 'G06: Ground Water'
    elif 'Group 7' in name:
        return 'G07: RO <20 (Fasting)'
    elif 'Group 8' in name:
        return 'G08: Kalyani (Fasting)'
    elif 'Group 9' in name:
        return 'G09: BIS Standard'
    elif 'Group 10' in name:
        return 'G10: Ground (Fasting)'
    elif 'Group 11' in name:
        return 'G11: RO 125-150 (Fasting)'
    return name

# Function to sort group names properly (numerically)
def sort_groups(group_list):
    """Sort group names in proper numerical order (G01, G02, ..., G11)"""
    return sorted(group_list)

df_clean['group_short'] = df_clean['water_group'].apply(shorten_group_name)
fcr_summary['group_short'] = fcr_summary['water_group'].apply(shorten_group_name)

print("\nWater group mapping:")
for original, short in df_clean[['water_group', 'group_short']].drop_duplicates().values:
    print(f"  {short}")

# ============================================================================
# STEP 3: Create 3-Panel Figure
# ============================================================================
print("\n" + "="*80)
print("STEP 3: Creating visualizations")
print("="*80)

fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3, height_ratios=[1.2, 1, 1])

# ============================================================================
# PANEL A: Line Plot - FCR Over Time (All Groups)
# ============================================================================
print("\nCreating Panel A: Line plot...")
ax1 = fig.add_subplot(gs[0, :])

# Plot each water group (in proper numerical order)
for group in sort_groups(df_clean['group_short'].unique()):
    group_data = fcr_summary[fcr_summary['group_short'] == group]
    
    ax1.plot(group_data['week'], 
             group_data['fcr_median'], 
             marker='o', 
             linewidth=2.5, 
             markersize=6,
             label=group,
             alpha=0.8)
    
    # Add confidence bands (using std)
    ax1.fill_between(group_data['week'],
                      group_data['fcr_median'] - group_data['fcr_std'],
                      group_data['fcr_median'] + group_data['fcr_std'],
                      alpha=0.15)

ax1.set_xlabel('Week', fontsize=12, fontweight='bold')
ax1.set_ylabel('Feed Conversion Ratio (FCR)', fontsize=12, fontweight='bold')
ax1.set_title('Panel A: FCR Over Time by Water Group\n(Median ± SD, excluding weeks with <2g weight gain)', 
              fontsize=14, fontweight='bold', pad=20)
ax1.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9, framealpha=0.9)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(range(1, 16))

# Add horizontal line at FCR=10 as reference
ax1.axhline(y=10, color='red', linestyle='--', alpha=0.3, linewidth=1, label='FCR=10 (reference)')

# ============================================================================
# PANEL B: Box Plots by Phase
# ============================================================================
print("Creating Panel B: Box plots by phase...")
ax2 = fig.add_subplot(gs[1, :])

# Prepare data for box plot
phases_order = ['Early (1-5)', 'Middle (6-10)', 'Late (11-15)']
groups_order = sort_groups(df_clean['group_short'].unique())

# Create positions for boxes
n_groups = len(groups_order)
n_phases = len(phases_order)
positions = []
labels = []
data_to_plot = []
colors = []

# Get color palette
palette = sns.color_palette("husl", n_groups)
group_colors = {group: palette[i] for i, group in enumerate(groups_order)}

for phase_idx, phase in enumerate(phases_order):
    for group_idx, group in enumerate(groups_order):
        group_phase_data = df_clean[(df_clean['group_short'] == group) & 
                                     (df_clean['phase'] == phase)]['fcr']
        
        if len(group_phase_data) > 0:
            pos = phase_idx * (n_groups + 1) + group_idx
            positions.append(pos)
            data_to_plot.append(group_phase_data)
            colors.append(group_colors[group])
            
            if phase_idx == 0:
                labels.append(group)

# Create box plot
bp = ax2.boxplot(data_to_plot, 
                  positions=positions,
                  widths=0.6,
                  patch_artist=True,
                  showfliers=True,
                  flierprops=dict(marker='o', markersize=3, alpha=0.3))

# Color the boxes
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add phase separators and labels
for phase_idx, phase in enumerate(phases_order):
    center_pos = phase_idx * (n_groups + 1) + (n_groups - 1) / 2
    ax2.text(center_pos, ax2.get_ylim()[1] * 0.95, phase, 
             ha='center', va='top', fontsize=11, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    if phase_idx < n_phases - 1:
        separator_pos = (phase_idx + 1) * (n_groups + 1) - 0.5
        ax2.axvline(x=separator_pos, color='gray', linestyle='--', alpha=0.5, linewidth=1.5)

ax2.set_ylabel('Feed Conversion Ratio (FCR)', fontsize=12, fontweight='bold')
ax2.set_title('Panel B: FCR Distribution by Growth Phase', 
              fontsize=14, fontweight='bold', pad=20)
ax2.set_xticks([])
ax2.grid(True, alpha=0.3, axis='y')

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=group_colors[group], alpha=0.7, label=group) 
                   for group in groups_order]
ax2.legend(handles=legend_elements, bbox_to_anchor=(1.01, 1), loc='upper left', 
           fontsize=8, framealpha=0.9)

# ============================================================================
# PANEL C: Heatmap - FCR by Group and Week
# ============================================================================
print("Creating Panel C: Heatmap...")
ax3 = fig.add_subplot(gs[2, :])

# Prepare heatmap data
heatmap_data = df_clean.groupby(['group_short', 'week'])['fcr'].median().reset_index()
heatmap_pivot = heatmap_data.pivot(index='group_short', columns='week', values='fcr')

# Sort groups in proper order (G01 -> G11)
heatmap_pivot = heatmap_pivot.reindex(sort_groups(heatmap_pivot.index))

# Create heatmap
sns.heatmap(heatmap_pivot, 
            annot=True, 
            fmt='.1f', 
            cmap='RdYlGn_r',  # Red (high FCR/bad) to Green (low FCR/good)
            linewidths=0.5,
            cbar_kws={'label': 'Median FCR'},
            ax=ax3,
            vmin=0,
            vmax=15)

ax3.set_xlabel('Week', fontsize=12, fontweight='bold')
ax3.set_ylabel('Water Group', fontsize=12, fontweight='bold')
ax3.set_title('Panel C: FCR Heatmap - Median FCR by Group and Week\n(Green = Efficient, Red = Inefficient)', 
              fontsize=14, fontweight='bold', pad=20)

# Rotate y-axis labels for better readability
ax3.set_yticklabels(ax3.get_yticklabels(), rotation=0, fontsize=9)

# ============================================================================
# Save Figure
# ============================================================================
print("\nSaving figure...")
fig.suptitle('Feed Conversion Ratio (FCR) Analysis Across Water Groups', 
             fontsize=16, fontweight='bold', y=0.995)

output_path = output_dir / 'fcr_analysis_3panel.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✓ Saved: {output_path}")

plt.close()

# ============================================================================
# STEP 4: Create Validation Plot (Efficiency Score vs FCR)
# ============================================================================
print("\n" + "="*80)
print("STEP 4: Creating validation plot (Efficiency Score)")
print("="*80)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: FCR over time
for group in sort_groups(df_clean['group_short'].unique()):
    group_data = df_clean[df_clean['group_short'] == group].groupby('week')['fcr'].median()
    ax1.plot(group_data.index, group_data.values, marker='o', label=group, linewidth=2, alpha=0.8)

ax1.set_xlabel('Week', fontsize=12, fontweight='bold')
ax1.set_ylabel('FCR (Feed Intake / Weight Gain)', fontsize=12, fontweight='bold')
ax1.set_title('FCR Over Time', fontsize=13, fontweight='bold')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(range(1, 16))

# Plot 2: Efficiency Score over time (inverse metric)
for group in sort_groups(df_clean['group_short'].unique()):
    group_data = df_clean[df_clean['group_short'] == group].groupby('week')['efficiency_score'].median()
    ax2.plot(group_data.index, group_data.values, marker='o', label=group, linewidth=2, alpha=0.8)

ax2.set_xlabel('Week', fontsize=12, fontweight='bold')
ax2.set_ylabel('Efficiency Score (Weight Gain / Feed Intake × 100)', fontsize=12, fontweight='bold')
ax2.set_title('Efficiency Score Over Time (Validation)', fontsize=13, fontweight='bold')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(1, 16))

plt.tight_layout()

output_path_validation = output_dir / 'fcr_validation_efficiency_score.png'
plt.savefig(output_path_validation, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✓ Saved: {output_path_validation}")

plt.close()

# ============================================================================
# STEP 5: Generate Summary Statistics
# ============================================================================
print("\n" + "="*80)
print("STEP 5: Summary Statistics")
print("="*80)

# Overall FCR by water group
print("\n--- Overall FCR by Water Group (All Weeks) ---")
fcr_by_group = df_clean.groupby('group_short')['fcr'].agg(['mean', 'median', 'std', 'min', 'max', 'count'])
# Sort by group number, not by median
fcr_by_group = fcr_by_group.reindex(sort_groups(fcr_by_group.index))
print(fcr_by_group.round(2))

# FCR by phase
print("\n--- FCR by Phase ---")
fcr_by_phase = df_clean.groupby('phase')['fcr'].agg(['mean', 'median', 'std', 'count'])
print(fcr_by_phase.round(2))

# Best and worst performing groups (sorted by performance)
print("\n--- Best Performing Groups (Lowest Median FCR, sorted by performance) ---")
fcr_by_group_sorted = fcr_by_group.sort_values('median')
print(fcr_by_group_sorted.head(3)[['median', 'mean', 'std']])

print("\n--- Worst Performing Groups (Highest Median FCR, sorted by performance) ---")
print(fcr_by_group_sorted.tail(3)[['median', 'mean', 'std']])

# FCR deterioration analysis (compare early vs late)
print("\n--- FCR Deterioration (Early vs Late Phases) ---")
deterioration = df_clean.groupby(['group_short', 'phase'])['fcr'].median().unstack()
deterioration['deterioration'] = deterioration['Late (11-15)'] - deterioration['Early (1-5)']
# Sort by group number
deterioration = deterioration.reindex(sort_groups(deterioration.index))
print(deterioration.round(2))

# Save statistics to CSV
print("\n" + "="*80)
print("Saving summary statistics to CSV...")
print("="*80)

fcr_by_group.to_csv(output_dir / 'fcr_by_group_summary.csv')
fcr_by_phase.to_csv(output_dir / 'fcr_by_phase_summary.csv')
deterioration.to_csv(output_dir / 'fcr_deterioration_analysis.csv')

print(f"✓ Saved: fcr_by_group_summary.csv")
print(f"✓ Saved: fcr_by_phase_summary.csv")
print(f"✓ Saved: fcr_deterioration_analysis.csv")

print("\n" + "="*80)
print("FCR ANALYSIS COMPLETE!")
print("="*80)
print(f"\nGenerated files:")
print(f"  1. fcr_analysis_3panel.png - Main 3-panel visualization")
print(f"  2. fcr_validation_efficiency_score.png - Validation plot")
print(f"  3. fcr_by_group_summary.csv - Overall statistics by group")
print(f"  4. fcr_by_phase_summary.csv - Statistics by growth phase")
print(f"  5. fcr_deterioration_analysis.csv - Early vs late comparison")
print("\n" + "="*80)

Data loaded: 1650 rows
Unique rats: 110
Water groups: 11

STEP 1: Calculating FCR

Original rows: 1650
After excluding weight_gain < 2.0g: 1378
Rows excluded: 272 (16.5%)

--- FCR Statistics by Week (Clean Data) ---
           mean     median       std  count
week                                       
1      0.428954   0.392081  0.196089    110
2      0.678729   0.637647  0.274081    110
3      0.813185   0.788727  0.260183    110
4      1.039238   1.007515  0.300723    110
5      1.062481   0.990339  0.319154    110
6      1.480445   1.395083  0.437890    110
7      1.683537   1.617308  0.591903    110
8      1.973271   1.887000  0.730754    109
9      2.811722   2.525500  1.233493    104
10     3.137743   2.793000  1.263930     97
11     3.934022   3.724600  1.410472     85
12     5.069520   4.750750  2.226560     69
13     6.019098   5.940000  2.383722     61
14     8.350625   8.850000  3.130127     48
15    10.992381  11.000000  2.390748     35

--- Weeks Most Affected by <2g Excl